# [참고·심화] 클래스 설계 5원칙 S.O.L.I.D (학생용)

> 이 단원은 **참고 사항**입니다. 외우기보다 "왜 이렇게 설계하는가"를 이해하는 것이 목표입니다.
> 출처: Samuel Oloruntoba, *S.O.L.I.D: The First 5 Principles of Object Oriented Design* (scotch.io, 현재 DigitalOcean Community로 이전됨). 원문은 PHP 도형 예제이며, 이 교안은 같은 흐름을 Python으로 옮겼습니다.
> 모든 예제는 Python 3.10에서 실행해 출력 결과를 확인했습니다.

**노트북 사용법**

- 위에서부터 차례로 셀을 실행하세요 (`Shift + Enter`).
- 예제 코드는 실행하기 전에 결과를 먼저 예측해 보세요.
- 실습 문제는 `# TODO` 부분을 채운 뒤, 바로 아래 **테스트 셀**을 실행해 `테스트 통과!`가 나오면 완료입니다.

## 학습 목표

- SOLID 다섯 원칙의 이름과 핵심 의미를 한 문장으로 설명할 수 있다.
- 각 원칙을 **위반한 코드**에서 어떤 문제가 생기는지 찾을 수 있다.
- `abc.ABC`, `@abstractmethod`로 인터페이스(추상 클래스)를 만들고 원칙에 맞게 개선할 수 있다.

## 0. 한눈에 보기

| 약자 | 원칙 | 한 줄 요약 | 위반 시 증상 |
| --- | --- | --- | --- |
| **S** | SRP: 단일 책임 원칙 (Single Responsibility Principle) | 클래스를 바꿀 이유는 **하나**뿐이어야 한다 | 한 클래스가 계산도, 출력도, 저장도 함 |
| **O** | OCP: 개방-폐쇄 원칙 (Open-Closed Principle) | **확장에는 열려** 있고 **수정에는 닫혀** 있어야 한다 | 기능 추가마다 `if/elif`를 늘림 |
| **L** | LSP: 리스코프 치환 원칙 (Liskov Substitution Principle) | 자식 객체는 부모 객체 자리에 **그대로 대체**될 수 있어야 한다 | 자식으로 바꾸면 결과가 이상해짐 |
| **I** | ISP: 인터페이스 분리 원칙 (Interface Segregation Principle) | 쓰지 않는 메서드 구현을 **강요하지 않는다** | `raise NotImplementedError`로 때우는 메서드 |
| **D** | DIP: 의존성 역전 원칙 (Dependency Inversion Principle) | 구체 클래스가 아니라 **추상화에 의존**한다 | 클래스 안에서 다른 구체 클래스를 직접 생성 |

**사전 지식: 파이썬의 "인터페이스"**

파이썬에는 Java의 `interface` 키워드가 없습니다. 대신 `abc` 모듈의 **추상 클래스**로 "이 메서드는 반드시 구현하라"는 약속을 만듭니다.

In [ ]:
from abc import ABC, abstractmethod

class Shape(ABC):
    @abstractmethod
    def area(self): ...      # 자식 클래스가 반드시 구현해야 하는 메서드

추상 메서드를 구현하지 않은 클래스는 객체를 만들 수 없습니다(`TypeError`). 이 단원의 예제는 이 기능을 계속 사용합니다.

## 1. S: 단일 책임 원칙 (SRP)

> "A class should have one and only one reason to change, meaning that a class should have only one job."
> 클래스는 변경될 이유가 단 하나여야 한다. 즉, 하나의 일만 해야 한다.

**문제 상황**: `AreaCalculator`가 넓이 합을 **계산**하면서 HTML로 **출력**까지 합니다. 출력 형식(JSON, CSV 등)이 추가될 때마다 계산 클래스를 고쳐야 합니다. 바뀌는 이유가 두 가지(계산 로직, 출력 형식)인 것입니다.

**해결**: 계산 클래스와 출력 클래스를 나눕니다.

In [ ]:
import json
import math

# [위반]: 계산과 출력 형식을 한 클래스가 모두 책임
class AreaCalculatorBad:
    def __init__(self, shapes):
        self.shapes = shapes

    def sum(self):
        return sum(s.area() for s in self.shapes)

    def output_html(self):
        return f"<h1>넓이 합: {self.sum():.2f}</h1>"
    # JSON, CSV가 필요해지면 계산 클래스를 계속 수정해야 함


# [개선]: 계산 책임 / 출력 책임 분리
class Square:
    def __init__(self, length):
        self.length = length
    def area(self):
        return self.length ** 2

class Circle:
    def __init__(self, radius):
        self.radius = radius
    def area(self):
        return math.pi * self.radius ** 2

class AreaCalculator:                 # 책임 1: 넓이 합 계산
    def __init__(self, shapes):
        self.shapes = shapes
    def sum(self):
        return sum(s.area() for s in self.shapes)

class SumCalculatorOutputter:        # 책임 2: 결과 출력 형식
    def __init__(self, calculator):
        self.calculator = calculator
    def html(self):
        return f"<h1>넓이 합: {self.calculator.sum():.2f}</h1>"
    def json(self):
        return json.dumps({"sum": round(self.calculator.sum(), 2)})

calc = AreaCalculator([Circle(2), Square(5), Square(6)])
out = SumCalculatorOutputter(calc)
print(out.html())
print(out.json())

**포인트**

- "책임"이란 **변경의 이유**다. 메서드가 하나여야 한다는 뜻이 아니다.
- 출력 형식을 추가해도 `AreaCalculator`는 건드리지 않는다.

## 2. O: 개방-폐쇄 원칙 (OCP)

> "Objects or entities should be open for extension but closed for modification."
> 객체나 개체는 확장에는 열려 있고, 수정에는 닫혀 있어야 한다.

**문제 상황**: 계산기가 도형 종류를 `if/elif`로 구분합니다. 삼각형을 추가하려면 **이미 검증된 계산기 코드를 수정**해야 합니다.

**해결**: 넓이 계산 책임을 각 도형으로 옮기고, 공통 인터페이스 `Shape`를 둡니다. 새 도형은 **클래스를 추가**하는 것만으로 지원됩니다.

In [ ]:
import math
from abc import ABC, abstractmethod

# [위반]: 새 도형이 생길 때마다 계산기 코드를 수정해야 함
class AreaCalculatorBad:
    def __init__(self, shapes):
        self.shapes = shapes
    def sum(self):
        total = 0
        for s in self.shapes:
            if s["type"] == "square":
                total += s["length"] ** 2
            elif s["type"] == "circle":
                total += math.pi * s["radius"] ** 2
            # 삼각형 추가? -> elif를 또 써야 함 (수정에 열려 있음)
        return total


# [개선]: 도형이 스스로 넓이를 계산 -> 계산기는 수정 없이 확장
class Shape(ABC):
    @abstractmethod
    def area(self) -> float: ...

class Square(Shape):
    def __init__(self, length):
        self.length = length
    def area(self):
        return self.length ** 2

class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius
    def area(self):
        return math.pi * self.radius ** 2

class Triangle(Shape):               # 새 도형 추가: 기존 코드 수정 0줄
    def __init__(self, base, height):
        self.base, self.height = base, height
    def area(self):
        return self.base * self.height / 2

class AreaCalculator:
    def __init__(self, shapes):
        self.shapes = shapes
    def sum(self):
        return sum(s.area() for s in self.shapes)

print(round(AreaCalculator([Square(2), Circle(1), Triangle(4, 3)]).sum(), 2))

try:
    class Hexagon(Shape):            # area()를 구현하지 않으면?
        pass
    Hexagon()
except TypeError as e:
    print("TypeError:", e)

**포인트**

- `Triangle`을 추가할 때 `AreaCalculator`는 한 줄도 바뀌지 않았다. 이것이 **다형성**을 이용한 확장이다.
- `@abstractmethod` 덕분에 `area()`를 빠뜨린 도형은 객체 생성 시점에 바로 오류가 난다.
- 앞 단원의 연산자 중복 정의도 같은 원리다. `+`(내장 코드)를 수정하지 않고 `__add__`만 구현해 동작을 확장했다.

## 3. L: 리스코프 치환 원칙 (LSP)

> "Let q(x) be a property provable about objects of x of type T. Then q(y) should be provable for objects y of type S where S is a subtype of T."
> 쉽게 말해, 부모 타입을 쓰는 코드에 자식 타입 객체를 넣어도 **프로그램이 올바르게 동작**해야 한다.

**문제 상황 (고전 예시)**: 수학적으로 정사각형은 직사각형이므로 `Square(Rectangle)` 상속이 자연스러워 보입니다. 그런데 "너비 5, 높이 4로 바꾸면 넓이는 20"이라는 부모의 약속을 자식이 깨뜨립니다.

**원문 예시**: `VolumeCalculator`가 `AreaCalculator`를 상속했다면 `sum()`은 부모와 **같은 타입(숫자)**을 반환해야 합니다. 리스트를 반환하면 부모를 쓰던 코드가 깨집니다.

In [ ]:
# [위반]: 정사각형은 직사각형이다(수학적으로)? -> 코드에서는 치환 불가
class Rectangle:
    def __init__(self, w, h):
        self._w, self._h = w, h
    def set_width(self, w):
        self._w = w
    def set_height(self, h):
        self._h = h
    def area(self):
        return self._w * self._h

class Square(Rectangle):
    def __init__(self, size):
        super().__init__(size, size)
    def set_width(self, w):            # 정사각형 유지를 위해 동작을 바꿈
        self._w = self._h = w
    def set_height(self, h):
        self._w = self._h = h

def resize_and_check(rect: Rectangle):
    rect.set_width(5)
    rect.set_height(4)
    return rect.area()                 # 사용하는 쪽은 20을 기대

print("Rectangle:", resize_and_check(Rectangle(1, 1)))
print("Square   :", resize_and_check(Square(1)))   # 16 -> 기대 깨짐


# [개선]: 공통 상위 타입은 '넓이를 가진다'까지만 약속
from abc import ABC, abstractmethod

class Shape(ABC):
    @abstractmethod
    def area(self): ...

class Rect(Shape):
    def __init__(self, w, h):
        self.w, self.h = w, h
    def area(self):
        return self.w * self.h

class Sq(Shape):
    def __init__(self, size):
        self.size = size
    def area(self):
        return self.size ** 2

# 원문 예제: 반환 타입이 같아야 치환 가능
class AreaCalculator:
    def __init__(self, shapes):
        self.shapes = shapes
    def sum(self):
        return sum(s.area() for s in self.shapes)        # 항상 숫자 반환

class VolumeCalculator(AreaCalculator):
    def sum(self):
        return super().sum() * 2                        # [O] 숫자 반환 유지
        # return [super().sum()]                        # [X] 리스트 반환 -> 부모 대체 불가

shapes = [Rect(5, 4), Sq(4)]
for calc in (AreaCalculator(shapes), VolumeCalculator(shapes)):
    print(type(calc).__name__, "->", f"{calc.sum():,}")  # 둘 다 숫자로 다룸

**포인트**

- "**is-a 관계**"는 현실의 분류가 아니라 **행동(약속)이 같은지**로 판단한다.
- 자식은 부모 메서드의 **매개변수 조건을 더 엄격하게** 만들거나 **반환 타입을 바꾸면** 안 된다.
- 상속이 어색하면 상속 대신 **공통 추상 타입**을 두거나 **조합(composition)**을 쓴다.

## 4. I: 인터페이스 분리 원칙 (ISP)

> "A client should never be forced to implement an interface that it doesn't use, or clients shouldn't be forced to depend on methods they do not use."
> 클라이언트는 사용하지 않는 인터페이스의 구현을 강요받아서는 안 된다.

**문제 상황**: 3차원 도형을 지원하려고 `ShapeInterface`에 `volume()`을 추가하면, 정사각형·원 같은 2차원 도형도 의미 없는 `volume()`을 억지로 구현해야 합니다.

**해결**: 인터페이스를 역할별로 **작게 쪼개고**, 클래스는 필요한 것만 골라 상속(다중 상속)합니다. 원문처럼 계산기가 사용할 공통 진입점 `ManageShapeInterface.calculate()`도 둡니다.

In [ ]:
from abc import ABC, abstractmethod

# [위반]: 모든 도형에 volume()을 강요
class ShapeInterfaceBad(ABC):
    @abstractmethod
    def area(self): ...
    @abstractmethod
    def volume(self): ...

class SquareBad(ShapeInterfaceBad):
    def __init__(self, l):
        self.l = l
    def area(self):
        return self.l ** 2
    def volume(self):                  # 2차원 도형인데 억지로 구현
        raise NotImplementedError("정사각형은 부피가 없음")


# [개선]: 작은 인터페이스로 분리 후 필요한 것만 조합
class ShapeInterface(ABC):
    @abstractmethod
    def area(self): ...

class ThreeDimensionalShapeInterface(ABC):
    @abstractmethod
    def volume(self): ...

class ManageShapeInterface(ABC):       # 계산기가 의존할 공통 인터페이스
    @abstractmethod
    def calculate(self): ...

class Square(ShapeInterface, ManageShapeInterface):
    def __init__(self, l):
        self.l = l
    def area(self):
        return self.l ** 2
    def calculate(self):
        return self.area()

class Cuboid(ShapeInterface, ThreeDimensionalShapeInterface, ManageShapeInterface):
    def __init__(self, w, h, d):
        self.w, self.h, self.d = w, h, d
    def area(self):                    # 겉넓이
        return 2 * (self.w * self.h + self.h * self.d + self.d * self.w)
    def volume(self):
        return self.w * self.h * self.d
    def calculate(self):
        return self.volume()

for s in [Square(3), Cuboid(2, 3, 4)]:
    print(type(s).__name__,
          "| 3D?", isinstance(s, ThreeDimensionalShapeInterface),
          "| calculate =", s.calculate())

**포인트**

- "큰 인터페이스 하나"보다 "작은 인터페이스 여러 개"가 낫다.
- `NotImplementedError`를 던지는 메서드가 보이면 ISP(그리고 LSP) 위반을 의심한다.

## 5. D: 의존성 역전 원칙 (DIP)

> "Entities must depend on abstractions, not on concretions. High-level modules must not depend on low-level modules, but they should depend on abstractions."
> 고수준 모듈은 저수준 모듈에 의존하면 안 되며, 둘 다 추상화에 의존해야 한다.

**문제 상황**: 비즈니스 로직(`PasswordReminder`, 고수준)이 특정 DB(`MySQLConnection`, 저수준)를 **직접 생성**합니다. DB를 바꾸거나 테스트용 가짜 DB를 넣으려면 비즈니스 로직을 수정해야 합니다.

**해결**: `DBConnectionInterface`라는 추상화를 만들고, 실제 객체는 **바깥에서 주입**(Dependency Injection)합니다.

In [ ]:
from abc import ABC, abstractmethod

# [위반]: 상위 모듈이 구체 클래스(MySQL)에 직접 의존
class MySQLConnectionBad:
    def connect(self):
        return "MySQL 연결"

class PasswordReminderBad:
    def __init__(self):
        self.db = MySQLConnectionBad()   # DB를 바꾸려면 이 클래스를 수정해야 함


# [개선]: 상위·하위 모듈 모두 추상화(인터페이스)에 의존
class DBConnectionInterface(ABC):
    @abstractmethod
    def connect(self) -> str: ...

class MySQLConnection(DBConnectionInterface):
    def connect(self):
        return "MySQL 연결"

class PostgreSQLConnection(DBConnectionInterface):
    def connect(self):
        return "PostgreSQL 연결"

class FakeConnection(DBConnectionInterface):      # 테스트용 가짜 DB
    def connect(self):
        return "테스트용 가짜 연결"

class PasswordReminder:
    def __init__(self, db: DBConnectionInterface):   # 의존성 주입(DI)
        self.db = db
    def remind(self, user):
        return f"[{self.db.connect()}] {user}님께 비밀번호 재설정 메일 발송"

for db in (MySQLConnection(), PostgreSQLConnection(), FakeConnection()):
    print(PasswordReminder(db).remind("kim"))

**포인트**

- 의존 방향이 `PasswordReminder → MySQL`에서 `PasswordReminder → 인터페이스 ← MySQL`로 **역전**된다.
- 생성자로 의존 객체를 받는 방식(생성자 주입)이 가장 흔하다.
- 테스트할 때 진짜 DB 대신 `FakeConnection`을 넣을 수 있다는 것이 큰 장점이다.

## 6. 파이썬다운 참고 사항

- **덕 타이핑**: 파이썬은 추상 클래스를 상속하지 않아도 같은 메서드만 있으면 동작한다("오리처럼 걷고 꽥꽥대면 오리"). `abc`는 **약속을 명시하고 실수를 일찍 잡기 위한 도구**다.
- **`typing.Protocol`** (Python 3.8+): 상속 없이 "이런 메서드를 가진 타입"을 표현할 수 있어 DIP·ISP에 자주 쓰인다.
- **과유불급**: 작은 스크립트에까지 SOLID를 엄격히 적용하면 클래스만 늘고 읽기 어려워진다. 변경이 잦은 부분, 여러 사람이 함께 쓰는 코드부터 적용한다.

## 7. 정리

| 원칙 | 핵심 질문 | 주요 도구 |
| --- | --- | --- |
| SRP | 이 클래스를 바꿀 이유가 둘 이상인가? | 클래스 분리 |
| OCP | 기능을 추가할 때 기존 코드를 고치는가? | 추상 클래스, 다형성 |
| LSP | 자식으로 바꿔 넣어도 결과가 같은가? | 올바른 상속 설계, 반환 타입 유지 |
| ISP | 쓰지 않는 메서드를 억지로 구현하는가? | 작은 인터페이스, 다중 상속 |
| DIP | 구체 클래스를 직접 생성하는가? | 추상화 + 의존성 주입 |

## 실습 문제

각 문제의 `# TODO`를 채우고, 아래 테스트 셀을 실행해 `테스트 통과!`가 나오는지 확인하세요.

### 실습 1 SRP: 책임 나누기

아래 `UserServiceBad`는 **이메일 검증, 저장, 메일 발송**을 모두 합니다. 다음 세 클래스로 책임을 나누고, `UserService`는 이들을 조합만 하도록 고치세요.

- `EmailValidator.is_valid(email)` → `"@"`가 있으면 `True`
- `UserRepository.save(name, email)` / `UserRepository.all()` → 저장된 `(name, email)` 리스트
- `WelcomeMailer.send(email)` → `"{email}로 환영 메일 발송"` 문자열 반환
- `UserService(validator, repo, mailer).register(name, email)` → 검증 실패 시 `ValueError`, 성공 시 메일 메시지 반환

```python
class UserServiceBad:
    def __init__(self):
        self.users = []
    def register(self, name, email):
        if "@" not in email:
            raise ValueError("잘못된 이메일")
        self.users.append((name, email))
        return f"{email}로 환영 메일 발송"
```

In [ ]:
class EmailValidator:
    pass  # TODO

class UserRepository:
    pass  # TODO

class WelcomeMailer:
    pass  # TODO

class UserService:
    pass  # TODO

In [ ]:
# 테스트: 그대로 실행하세요
repo = UserRepository()
service = UserService(EmailValidator(), repo, WelcomeMailer())
print(service.register("kim", "kim@example.com"))
assert repo.all() == [("kim", "kim@example.com")]
try:
    service.register("lee", "lee-at-example.com")
    raise AssertionError("ValueError가 발생해야 합니다")
except ValueError:
    pass
assert len(repo.all()) == 1
print("테스트 통과!")

### 실습 2 OCP: `if/elif` 없애기

회원 등급별 할인율을 `if/elif`로 처리하던 코드를, **등급 클래스를 추가하는 것만으로** 확장되도록 바꾸세요.

- 추상 클래스 `Grade`에 추상 메서드 `rate()` (할인율, 0~1)
- `Basic`(0%), `Silver`(5%), `Gold`(10%) 구현
- `final_price(price, grade)` → `int(price * (1 - grade.rate()))`
- 마지막으로 `final_price`를 **수정하지 않고** `VIP`(20%)를 추가

```python
def final_price_bad(price, grade):
    if grade == "basic":
        return price
    elif grade == "silver":
        return int(price * 0.95)
    elif grade == "gold":
        return int(price * 0.9)
```

In [ ]:
from abc import ABC, abstractmethod

class Grade(ABC):
    pass  # TODO: 추상 메서드 rate()

# TODO: Basic, Silver, Gold, VIP

def final_price(price, grade):
    pass  # TODO

In [ ]:
# 테스트: 그대로 실행하세요
for g in [Basic(), Silver(), Gold(), VIP()]:
    print(type(g).__name__, final_price(10_000, g))
assert final_price(10_000, Basic()) == 10_000
assert final_price(10_000, Silver()) == 9_500
assert final_price(10_000, Gold()) == 9_000
assert final_price(10_000, VIP()) == 8_000
try:
    Grade()
    raise AssertionError("추상 클래스는 객체를 만들 수 없어야 합니다")
except TypeError:
    pass
print("테스트 통과!")

### 실습 3 DIP: 의존성 주입

`WeatherReporterBad`는 실제 날씨 API 클래스를 **직접 생성**해서 테스트하기 어렵습니다. 추상화에 의존하도록 바꾸세요.

- 추상 클래스 `WeatherSource`에 추상 메서드 `temperature()`
- `WeatherReporter(source).report()` → `"오늘 기온은 {t}도입니다"`
- 테스트용 `FakeWeather(t)` 구현 (생성자로 받은 기온을 그대로 반환)

```python
class RealWeatherAPI:
    def temperature(self):
        ...  # 실제로는 인터넷에 접속해 값을 가져옴

class WeatherReporterBad:
    def __init__(self):
        self.api = RealWeatherAPI()
    def report(self):
        return f"오늘 기온은 {self.api.temperature()}도입니다"
```

In [ ]:
from abc import ABC, abstractmethod

class WeatherSource(ABC):
    pass  # TODO

class FakeWeather(WeatherSource):
    pass  # TODO

class WeatherReporter:
    pass  # TODO

In [ ]:
# 테스트: 그대로 실행하세요
print(WeatherReporter(FakeWeather(25)).report())
assert WeatherReporter(FakeWeather(25)).report() == "오늘 기온은 25도입니다"
assert WeatherReporter(FakeWeather(-3)).report() == "오늘 기온은 -3도입니다"
assert isinstance(FakeWeather(0), WeatherSource)
print("테스트 통과!")

## 8. 확인 문제

**Q1.** 다음 클래스는 어떤 원칙을 위반하는가?

```python
class Report:
    def make(self): ...
    def save_to_file(self): ...
    def send_email(self): ...
```

**Q2.** 결제 수단이 추가될 때마다 `if method == "card": ... elif method == "kakao": ...`를 늘려야 한다. 어떤 원칙을 위반했고, 어떻게 고치면 좋은가?

**Q3.** `Bird` 클래스에 `fly()`가 있고 `Penguin(Bird)`의 `fly()`는 예외를 던진다. 어떤 원칙들을 위반하는가?

**Q4.** `class Service: def __init__(self): self.repo = MySQLRepo()` 코드를 DIP에 맞게 바꾸면?